# Store cutoff matchup values

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import os
from glob import glob

import histlib.matchup as match
from histlib.cstes import labels, zarr_dir, matchup_dir, var

/home1/datahome/mdemol/.miniconda3/envs/histenv2/lib/python3.9/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'distributed.scheduler.transition-log-length' has been deprecated; please use 'distributed.admin.low-level-log-length' instead
  warnings.warn(
/home1/datahome/mdemol/.miniconda3/envs/histenv2/lib/python3.9/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'distributed.comm.recent-messages-log-length' has been deprecated; please use 'distributed.admin.low-level-log-length' instead
  warnings.warn(


In [2]:
if True:
    from dask.distributed import Client
    from dask_jobqueue import PBSCluster

    # cluster = PBSCluster(cores=56, processes=28, walltime='04:00:00')
    # cluster = PBSCluster(cores=7, processes=7, walltime='04:00:00')
    cluster = PBSCluster(cores=10, processes=10, walltime="04:00:00")
    w = cluster.scale(jobs=2)
else:
    from dask.distributed import Client, LocalCluster

    cluster = LocalCluster()

client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: http://10.148.0.38:8787/status,
Dashboard: http://10.148.0.38:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.148.0.38:43998,Workers: 0
Dashboard: http://10.148.0.38:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# Big

In [3]:
l = "gps_Sentinel-3_A_2019"  # labels[30]

In [13]:
from histlib.matchup import add_adt_to_ds_data, add_low_pass_filter_to_data

cutoff = [3, 3.5]
chunk = 2000


def cutoff_matchup(l, T, cutoff):
    var = [
        "__site_matchup_indice",
        "obs",
        "box_x",
        "box_y",
        "alti_time_mid",
        "f",
        "drifter_vx",
        "drifter_vy",
        "drifter_acc_x_0",
        "drifter_acc_y_0",
        "drifter_coriolis_x_0",
        "drifter_coriolis_y_0",
    ]
    ds_data = xr.open_zarr(os.path.join(zarr_dir, f"{l}.zarr")).chunk(
        {"obs": chunk, "site_obs": -1}
    ).isel(obs = slice(150000, -1))
    ds_data = ds_data.where(ds_data.alti___distance < 2e5, drop=True).chunk(
        {"obs": chunk, "site_obs": -1}
    )
    ds_data = add_adt_to_ds_data(ds_data)
    drogue_status = ds_data.time < ds_data.drifter_drogue_lost_date.mean("site_obs")
    ds_data = ds_data[var]

    add_low_pass_filter_to_data(ds_data, T=T, cutoff=cutoff)

    # SELECT MATCHUP
    cc = [
        l
        for l in list(ds_data.coords)
        if l not in ["obs", "box_x", "box_y", "alti_time_mid"]
    ]
    ds_data = ds_data.reset_coords(cc)
    idx = ds_data.__site_matchup_indice.astype(int).compute()
    dsmf = (
        ds_data.sel(site_obs=idx)
        .isel(alti_time_mid=ds_data.dims["alti_time_mid"] // 2)
        .drop(["alti_time_mid", "alti_x_mid", "alti_y_mid"])[
            [v for v in ds_data if "drifter_acc_x_" in v or "drifter_coriolis_x_" in v]
        ]
    )
    for v in dsmf.variables:
        dsmf[v].attrs = ds_data[v].attrs
    return dsmf


def run_cutoff_matchup(l, T=12, cutoff=cutoff):
    ds = cutoff_matchup(l, T=T, cutoff=cutoff).chunk({"obs": 500}).persist()
    # store
    zarr = os.path.join(
        zarr_dir + "_ok", "cutoff_matchup", "cutoff_matchup_" + l + "_3_1.zarr"
    )
    ds.to_zarr(zarr, mode="w")
    print(f"matchup {l} storred in {zarr}")

In [14]:
run_cutoff_matchup(l, T=12, cutoff=cutoff)

284
284
matchup gps_Sentinel-3_A_2019 storred in /home/datawork-lops-oc/aponte/margot/historical_coloc_ok/cutoff_matchup/cutoff_matchup_gps_Sentinel-3_A_2019_3_1.zarr


In [15]:
ds1 = xr.open_dataset('/home/datawork-lops-oc/aponte/margot/historical_coloc_ok/cutoff_matchup/cutoff_matchup_gps_Sentinel-3_A_2019_3_1.zarr')
ds0 = xr.open_dataset('/home/datawork-lops-oc/aponte/margot/historical_coloc_ok/cutoff_matchup/cutoff_matchup_gps_Sentinel-3_A_2019_3_0.zarr')


In [20]:
ds1.to_zarr('/home/datawork-lops-oc/aponte/margot/historical_coloc_ok/cutoff_matchup/cutoff_matchup_gps_Sentinel-3_A_2019_3.zarr', mode
        ='')

In [21]:
ds = xr.open_dataset('/home/datawork-lops-oc/aponte/margot/historical_coloc_ok/cutoff_matchup/cutoff_matchup_gps_Sentinel-3_A_2019_3.zarr')

In [53]:
zarr = os.path.join(
    zarr_dir + "_ok", "cutoff_matchup", "cutoff_matchup_" + l + "_2.zarr"
)
ds = xr.open_dataset(zarr)
zarr = os.path.join(zarr_dir + "_ok", "cutoff_matchup", "cutoff_matchup_" + l + ".zarr")
ds2 = xr.open_dataset(zarr)
zarr = os.path.join(matchup_dir, f"matchup_{l}.zarr")

In [ ]:
dsm

In [52]:
for v in dsm:
    print(v, len(dsm[v].dropna("obs")))

es_cstrio_z15_alti_wd_x 36306
es_cstrio_z15_drifter_wd_x 36306
alti_ggx_adt_filtered 36279
alti_ggx_adt_filtered_ocean_tide 36279
alti_ggx_adt_filtered_ocean_tide_internal_tide 36279
alti_ggx_adt_filtered_ocean_tide_internal_tide_dac 36279
aviso_alti_ggx_adt 36200
aviso_drifter_ggx_adt 36145
drifter_acc_x_0 36302
drifter_coriolis_x_0 36306


In [57]:
merge = xr.merge([dsm, ds], join="inner").dropna("obs")

In [58]:
for v in merge:
    print(v, len(merge[v].dropna("obs")))

es_cstrio_z15_alti_wd_x 36085
es_cstrio_z15_drifter_wd_x 36085
alti_ggx_adt_filtered 36085
alti_ggx_adt_filtered_ocean_tide 36085
alti_ggx_adt_filtered_ocean_tide_internal_tide 36085
alti_ggx_adt_filtered_ocean_tide_internal_tide_dac 36085
aviso_alti_ggx_adt 36085
aviso_drifter_ggx_adt 36085
drifter_acc_x_0 36085
drifter_coriolis_x_0 36085
drifter_acc_x_15 36085
drifter_acc_x_25 36085
drifter_coriolis_x_15 36085
drifter_coriolis_x_25 36085


In [56]:
for v in merge:
    print(v, len(merge[v].dropna("obs")))

es_cstrio_z15_alti_wd_x 36306
es_cstrio_z15_drifter_wd_x 36306
alti_ggx_adt_filtered 36279
alti_ggx_adt_filtered_ocean_tide 36279
alti_ggx_adt_filtered_ocean_tide_internal_tide 36279
alti_ggx_adt_filtered_ocean_tide_internal_tide_dac 36279
aviso_alti_ggx_adt 36200
aviso_drifter_ggx_adt 36145
drifter_acc_x_0 36302
drifter_coriolis_x_0 36306
drifter_acc_x_15 36306
drifter_acc_x_25 36306
drifter_coriolis_x_15 36306
drifter_coriolis_x_25 36306


In [19]:
obsm = dsm.obs.values

In [20]:
obsds2 = ds2.obs.values

In [ ]:
obsm

In [13]:
ds2.drifter_acc_x_1.dropna(dim="obs")

<xarray.DataArray 'drifter_acc_x_1' (obs: 293330)>
array([ 7.708671e-06,  6.938183e-06,  4.184562e-07, ..., -2.625242e-06,
       -1.535585e-06,  3.019809e-06])
Coordinates:
  * obs      (obs) int64 0 1 2 3 4 5 ... 325266 325267 325268 325269 325270
Attributes:
    cutoff:       1
    description:  drifter's along track direction acceleration on the box fil...
    long_name:    $d_tu$
    units:        $m.s^{-2}$

In [15]:
ds.drifter_acc_x_15.dropna(dim="obs")

<xarray.DataArray 'drifter_acc_x_15' (obs: 293330)>
array([ 8.074178e-07, -1.358508e-06, -1.963585e-07, ..., -4.424513e-06,
       -7.748410e-07,  5.404292e-06])
Coordinates:
  * obs      (obs) int64 0 1 2 3 4 5 ... 325266 325267 325268 325269 325270
Attributes:
    cutoff:       1.5
    description:  drifter's along track direction acceleration on the box fil...
    long_name:    $d_tu$
    units:        $m.s^{-2}$

In [32]:
for l in labels:
    print(l)
    zarr = os.path.join(
        zarr_dir + "_ok", "cutoff_matchup", "cutoff_matchup_" + l + "_2.zarr"
    )
    ds = xr.open_dataset(zarr)
    zarr = os.path.join(
        zarr_dir + "_ok", "cutoff_matchup", "cutoff_matchup_" + l + ".zarr"
    )
    ds2 = xr.open_dataset(zarr)
    print(ds.dims["obs"], ds2.dims["obs"])

gps_Jason-3_2020
36694 36694
argos_Jason-3_2020
86 86
gps_SARAL_2020
37722 37722
argos_SARAL_2020
93 93
gps_Cryosat-2_2020
40027 40027
argos_Cryosat-2_2020
98 98
gps_Sentinel-3_A_2020
85718 85718
argos_Sentinel-3_A_2020
219 219
gps_Sentinel-3_B_2020
43312 43312
argos_Sentinel-3_B_2020
103 103
gps_Jason-3_2019
141640 141640
argos_Jason-3_2019
4218 4218
gps_SARAL_2019
133405 133405
argos_SARAL_2019
4159 4159
gps_Cryosat-2_2019
148210 148210
argos_Cryosat-2_2019
4439 4439
gps_Sentinel-3_A_2019
293330 293330
argos_Sentinel-3_A_2019
9730 9730
gps_Sentinel-3_B_2019
145749 145749
argos_Sentinel-3_B_2019
4847 4847
gps_Jason-3_2018
124069 124069
argos_Jason-3_2018
21429 21429
gps_SARAL_2018
126490 126490
argos_SARAL_2018
21471 21471
gps_Cryosat-2_2018
124112 124112
argos_Cryosat-2_2018
21275 21275
gps_Sentinel-3_A_2018
137229 137229
argos_Sentinel-3_A_2018
23237 23237
gps_Sentinel-3_B_2018
12574 12574
argos_Sentinel-3_B_2018
1182 1182
gps_Jason-3_2017
89700 89700
argos_Jason-3_2017
51008 51008


In [62]:
cluster.close()

NameError: name 'cluster' is not defined